# NLP Assignment — Spring 2025
**Student ID:** 23i-2548

---

## Part 1: Word Embeddings
- Vocabulary building & `word2idx.json`
- TF-IDF Matrix → `tfidf_matrix.npy`
- PPMI Matrix → `ppmi_matrix.npy`
- Word2Vec Skip-Gram → `embeddings_w2v.npy`
- Analogy tests & cosine similarity evaluation

In [ ]:
import collections
import json
import os
import re
import numpy as np
import torch
import torch.nn as nn
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 1.1 Load Corpus and Build Vocabulary

In [ ]:
with open("cleaned.txt", "r", encoding="utf-8") as f:
    docs_raw = f.read().splitlines()

# Filter headers
docs = [d for d in docs_raw if not re.match(r"^\\\[\\d+ \\\]", d.strip()) and d.strip()]
tokens_per_doc = [doc.split() for doc in docs]
all_tokens = [t for doc in tokens_per_doc for t in doc]

freq = collections.Counter(all_tokens)
vocab = ["<UNK>"] + [w for w, _ in freq.most_common(10000)]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

os.makedirs("embeddings", exist_ok=True)
with open("embeddings/word2idx.json", "w") as f:
    json.dump(word2idx, f)

### 1.2 TF-IDF Matrix

In [ ]:
N = len(docs)
V = len(vocab)

td = np.zeros((V, N), dtype=np.float32)
for j, doc in enumerate(tokens_per_doc):
    for w in doc:
        idx = word2idx.get(w, 0)
        td[idx, j] += 1

df = (td > 0).sum(axis=1)
idf = np.log(N / (1 + df))
tf = td / (td.sum(axis=0, keepdims=True) + 1e-9)
tfidf = tf * idf[:, None]

np.save("embeddings/tfidf_matrix.npy", tfidf)

### 1.3 PPMI Matrix

In [ ]:
k = 5
cooc = np.zeros((V, V), dtype=np.float32)

for doc in tokens_per_doc:
    idxs = [word2idx.get(w, 0) for w in doc]
    for i, ci in enumerate(idxs):
        for j in range(max(0, i-k), min(len(idxs), i+k+1)):
            if i != j:
                cooc[ci, idxs[j]] += 1

total = cooc.sum()
Pw = cooc.sum(axis=1) / total
Pww = cooc / total
with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.log2(Pww / (Pw[:, None] * Pw[None, :].reshape(1, -1) + 1e-9))
ppmi = np.maximum(0, pmi)

np.save("embeddings/ppmi_matrix.npy", ppmi)

### 1.4 t-SNE Plot & Neighbors Report

In [ ]:
def plot_tsne(matrix, labels, title):
    # Newer sklearn might not use n_iter or it might be renamed to max_iter
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(matrix)
    
    plt.figure(figsize=(12, 10))
    plt.scatter(coords[:, 0], coords[:, 1], alpha=0.6)
    for i, label in enumerate(labels):
        plt.annotate(label, (coords[i, 0], coords[i, 1]), fontsize=9)
    plt.title(title)
    plt.show()

# Plot top 200 frequent words using PPMI
top_200_items = freq.most_common(200)
top_200_words = [w for w, _ in top_200_items if w in word2idx]
top_200_idxs = [word2idx[w] for w in top_200_words]

plot_tsne(ppmi[top_200_idxs], top_200_words, "t-SNE of Top 200 Tokens (PPMI)")

# Report neighbors
queries = ["کرکٹ", "پاکستان", "حکومت", "عدالت", "لاہور", "اسلام", "معیشت", "سیاست", "کھیل", "فلم"]
for q in queries:
    if q in word2idx:
        q_idx = word2idx[q]
        sims = ppmi[q_idx]
        top_n = np.argsort(sims)[-6:][::-1]
        neighbors = [idx2word[i] for i in top_n if i != q_idx][:5]
        print(f"{q}: {', '.join(neighbors)}")

---
## Part 2: POS Tagging & Named Entity Recognition
- Manual annotation of 500 sentences (POS + NER)
- Rule-based POS tagger
- Gazetteer-based NER
- Evaluation with conlleval

In [ ]:
# Part 2 — POS Tagging & NER
# TODO: annotation, rule-based tagger, gazetteer NER, conlleval scoring

---
## Part 3: Transformer Encoder
- `ScaledDotProductAttention`
- `MultiHeadAttention`
- `PositionwiseFFN`
- `EncoderBlock`
- Sinusoidal Positional Encoding (register_buffer)
- [CLS] token classification head

In [ ]:
# Part 3 — Transformer Encoder (custom, no nn.Transformer)
# TODO: ScaledDotProductAttention, MultiHeadAttention, PositionwiseFFN, EncoderBlock

---
## Part 4: CRF with Viterbi Inference
- CRF layer
- Viterbi decoding (no greedy)
- NER evaluation with conlleval

In [ ]:
# Part 4 — CRF + Viterbi
# TODO: CRF model, Viterbi inference, conlleval NER scoring